In [1]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./raw_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./raw_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "./cleaned_data/gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "./cleaned_data/leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


In [2]:
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding='utf-8-sig')

In [ ]:
df_gmv.head()

In [4]:
print(df_gmv["TEAM"].unique())

<StringArray>
['HCM', 'In-house', 'In-house 2', 'Linh Dam Store']
Length: 4, dtype: str


In [ ]:
unique_prefix = (
    df_leads["TVTS"]
    .dropna()
    .str.split("-", n=1)
    .str[0]
    .str.strip()
    .unique()
)

print(unique_prefix)

['HN' 'HN2' 'HCM' 'OFF' 'HuyenKTT' 'LyBP' 'AnhVP' 'kho' 'Phuong' 'tr'
 'IND' 'TamNTT' 'TrangDT' 'LinhNTT' 'ThinhND']


In [15]:
# Drop NaN values to avoid errors during string operations
tvts_series = df_leads["TVTS"].dropna()

# Strip spaces and filter the Series for values that start with 'OFF'
off_values = tvts_series[tvts_series.str.strip().str.startswith("OFF")]

# If you want to see all occurrences
print(len(off_values))

# If you only want the unique 'OFF - something' string values
unique_off_values = off_values.unique()
print(unique_off_values)

50
<StringArray>
['OFF - Julia']
Length: 1, dtype: str


In [ ]:
import pandas as pd
import numpy as np

# 1. Define the exact mapping for the 4 TEAM cases
team_mapping = {
    "In-house": "HN",
    "In-house 2": "HN2",
    "HCM": "HCM",
    "Linh Dam Store": "OFF"
}

# 2. Function to automatically format Vietnamese names
def format_vietnamese_name(name):
    # Handle empty or NaN values
    if pd.isna(name):
        return ""
    
    # Strip leading/trailing spaces
    name = str(name).strip()
    parts = name.split()
    
    # If the name has spaces (e.g., "Cao Thi Lua"), convert to "LuaCT"
    if len(parts) > 1:
        # Take the last word as the main name
        first_name = parts[-1]
        # Get the first letter of all preceding words and uppercase them
        initials = "".join([p[0].upper() for p in parts[:-1]])
        return f"{first_name}{initials}"
    
    # If it's already a single string without spaces (e.g., "LinhNT"), return as is
    return name

# 3. Create the new column for abbreviated Sales names ONLY
df_gmv["Sales_Abbr"] = df_gmv["Sales"].apply(format_vietnamese_name)

# 4. Apply the team mapping to get the corresponding prefixes
prefixes = df_gmv["TEAM"].map(team_mapping)

# 5. Define conditions and choices for the 'Sales_infor' column
conditions = [
    # Condition 1: If TEAM is Linh Dam Store, hardcode the value to 'OFF - Julia'
    df_gmv["TEAM"] == "Linh Dam Store",
    
    # Condition 2: For other mapped teams, combine prefix and the abbreviated name
    prefixes.notna()
]

choices = [
    # Output for Condition 1
    "OFF - Julia",
    
    # Output for Condition 2
    prefixes + " - " + df_gmv["Sales_Abbr"]
]

# 6. Apply np.select to create the 'Sales_infor' column based on the conditions
# Teams not in the mapping will get NaN
df_gmv["Sales_infor"] = np.select(conditions, choices, default=np.nan)

# Print a sample to verify both new columns
print(df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]])

# Export to CSV
df_gmv.to_csv('./cleaned_data/gmv_clean_sales_infor.csv', encoding='utf-8-sig', index=False)

           TEAM               Sales Sales_Abbr     Sales_infor
0           HCM    Nguyen Minh Phat     PhatNM    HCM - PhatNM
1           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
2           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
3           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
4           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
..          ...                 ...        ...             ...
458    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
459    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
460    In-house   Nguyen Kieu Trang    TrangNK    HN - TrangNK
461  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT
462  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT

[463 rows x 4 columns]


In [17]:
# Group the dataframe by 'TEAM' and extract the first 3 rows of each group
# We only select the relevant columns for a cleaner output
sample_check = df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]].groupby("TEAM").head(3)

# Display the result to verify the applied logic
print(sample_check)

               TEAM               Sales Sales_Abbr    Sales_infor
0               HCM    Nguyen Minh Phat     PhatNM   HCM - PhatNM
1               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
2               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
19         In-house         Cao Thi Lua      LuaCT     HN - LuaCT
20         In-house        Le Thi Tuyet    TuyetLT   HN - TuyetLT
21         In-house   Pham Thi Tam Thuy    ThuyPTT   HN - ThuyPTT
34       In-house 2         Le Thi Tram     TramLT   HN2 - TramLT
35       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
48       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
56   Linh Dam Store   Ngo Thi Thuy Linh    LinhNTT    OFF - Julia
123  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia
127  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia
